In [ ]:
!pip install openai requests python-dotenv pandas

In [1]:
import os
from getpass import getpass
from openai import OpenAI
import requests
import json
import time
import pandas as pd
from typing import Optional

# ========== 交互输入配置 ==========
print("=== Dify 电商客服评测 - 配置 ===")

# Dify 应用配置（不变）
DIFY_API_URL = input("Dify 应用 API URL（默认 http://localhost/v1/chat-messages）: ").strip()
if not DIFY_API_URL:
    DIFY_API_URL = "http://localhost/v1/chat-messages"
DIFY_API_KEY = getpass("Dify 应用 API Key: ")

# 阿里云百炼配置（替换 DeepSeek）
ALIYUN_API_KEY = getpass("阿里云百炼 API Key: ")

# 阿里云百炼 OpenAI 兼容客户端
# 官方明确：base_url = "https://dashscope.aliyuncs.com/compatible-mode/v1"
judge_client = OpenAI(
    api_key=ALIYUN_API_KEY,
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)

# 裁判模型：默认 qwen-plus（阿里云官方推荐，OpenAI 兼容接口支持）
JUDGE_MODEL = "qwen-plus"

print(f"\n✅ 配置完成")
print(f"   Dify URL: {DIFY_API_URL}")
print(f"   裁判模型: {JUDGE_MODEL}（阿里云百炼）")
print(f"   兼容端点: https://dashscope.aliyuncs.com/compatible-mode/v1")

=== Dify 电商客服评测 - 配置 ===


Dify 应用 API URL（默认 http://localhost/v1/chat-messages）:  
Dify 应用 API Key:  ········
阿里云百炼 API Key:  ········



✅ 配置完成
   Dify URL: http://localhost/v1/chat-messages
   裁判模型: qwen-plus（阿里云百炼）
   兼容端点: https://dashscope.aliyuncs.com/compatible-mode/v1


In [2]:
# ========== 评测集（18 条精选用例）============
eval_cases = [
    {"case_id":"TC-CONSULT-001","category":"商品咨询","difficulty":"简单","user_query":"索尼WF-1000XM5蓝牙耳机的降噪效果怎么样？","expected_intent":"consult","expected_branch":"商品咨询","expected_answer_contains":["降噪深度","42dB"]},
    {"case_id":"TC-CONSULT-002","category":"商品咨询","difficulty":"困难","user_query":"蓝牙耳机怎么选？有什么推荐吗？","expected_intent":"consult","expected_branch":"商品咨询","expected_answer_contains":["暂未收录","人工客服"]},
    {"case_id":"TC-ORDER-001","category":"订单查询","difficulty":"简单","user_query":"查订单 31786428571","expected_intent":"query_order","expected_branch":"订单查询","expected_answer_contains":["订单号","31786428571"]},
    {"case_id":"TC-ORDER-002","category":"订单查询","difficulty":"中等","user_query":"我的订单到哪了？","expected_intent":"query_order","expected_branch":"订单查询","expected_answer_contains":["物流","订单"]},
    {"case_id":"TC-REFUND-001","category":"退款换货","difficulty":"简单","user_query":"我要退订单 31786428572","expected_intent":"refund","expected_branch":"退款换货","expected_answer_contains":["退货","退款"]},
    {"case_id":"TC-REFUND-002","category":"退款换货","difficulty":"中等","user_query":"我要退款，你们这破店骗人","expected_intent":"refund","expected_branch":"退款换货","expected_answer_contains":["退款","7天"]},
    {"case_id":"TC-REFUND-003","category":"退款换货","difficulty":"困难","user_query":"退订单 31786428571","expected_intent":"refund","expected_branch":"退款换货","expected_answer_contains":["超过7天","不符合"]},
    {"case_id":"TC-COMPLAINT-001","category":"投诉","difficulty":"中等","user_query":"快递延迟了两天还没到，我很着急","expected_intent":"complaint","expected_branch":"投诉","expected_answer_contains":["抱歉","补偿"]},
    {"case_id":"TC-COMPLAINT-002","category":"投诉","difficulty":"简单","user_query":"我要投诉，我现在情绪很激动？","expected_intent":"complaint","expected_branch":"投诉","expected_answer_contains":["抱歉","投诉"]},
    {"case_id":"TC-COMPLAINT-003","category":"投诉","difficulty":"中等","user_query":"这已经是第三次投诉了，态度太差了！","expected_intent":"complaint","expected_branch":"投诉","expected_answer_contains":["升级","主管"]},
    {"case_id":"TC-COMPLAINT-004","category":"投诉","difficulty":"困难","user_query":"用了之后过敏，要曝光到网上！","expected_intent":"complaint","expected_branch":"投诉","expected_answer_contains":["升级","1小时"]},
    {"case_id":"TC-OTHER-001","category":"其他","difficulty":"简单","user_query":"你好","expected_intent":"other","expected_branch":"其他","expected_answer_contains":["您好","客服小商"]},
    {"case_id":"TC-OTHER-002","category":"其他","difficulty":"中等","user_query":"今天天气怎么样？","expected_intent":"other","expected_branch":"其他","expected_answer_contains":["电商","无法回答"]},
    {"case_id":"TC-OTHER-003","category":"其他","difficulty":"困难","user_query":"asdfghjkl","expected_intent":"other","expected_branch":"其他","expected_answer_contains":["抱歉","重新表述"]},
    {"case_id":"TC-OTHER-004","category":"其他","difficulty":"简单","user_query":"转人工","expected_intent":"other","expected_branch":"其他","expected_answer_contains":["转接","人工"]},
    {"case_id":"TC-BOUND-001","category":"边界混淆","difficulty":"困难","user_query":"我要投诉，我现在情绪很激动？","expected_intent":"complaint","expected_branch":"投诉","expected_answer_contains":["投诉"]},
    {"case_id":"TC-BOUND-002","category":"边界混淆","difficulty":"困难","user_query":"我要退款，你们这破店骗人","expected_intent":"refund","expected_branch":"退款换货","expected_answer_contains":["退款"]},
    {"case_id":"TC-BOUND-003","category":"边界混淆","difficulty":"困难","user_query":"订单怎么还没到，太慢了","expected_intent":"query_order","expected_branch":"订单查询","expected_answer_contains":["订单"]},
]

print(f"✅ 评测集加载完成：{len(eval_cases)} 条用例")

✅ 评测集加载完成：18 条用例


In [3]:
# ========== 调用 Dify 客服应用 ==========
def call_dify_app(user_query: str, conversation_id: Optional[str] = None):
    """调用 Dify Chatflow 电商客服应用"""
    headers = {
        "Authorization": f"Bearer {DIFY_API_KEY}",
        "Content-Type": "application/json"
    }
    data = {
        "inputs": {},
        "query": user_query,
        "user": "eval-user",
        "response_mode": "blocking"
    }
    if conversation_id:
        data["conversation_id"] = conversation_id
    
    try:
        resp = requests.post(DIFY_API_URL, headers=headers, json=data, timeout=30)
        resp.raise_for_status()
        result = resp.json()
        return result.get("answer", ""), result.get("conversation_id", "")
    except Exception as e:
        return f"[ERROR] {str(e)}", conversation_id


# ========== DeepSeek 当裁判打分 ==========
def llm_judge(case: dict, actual_response: str):
    """用 DeepSeek 评估客服回复质量"""
    judge_prompt = f"""你是一名严格的电商客服质量评审专家。请评估以下客服回复的质量。

【用户问题】{case['user_query']}
【期望意图】{case['expected_intent']}
【期望分支】{case['expected_branch']}
【期望答案应包含的关键要点】{', '.join(case.get('expected_answer_contains', []))}

【实际客服回复】{actual_response}

请从以下维度严格打分（0-5分），并输出 JSON 格式：
{{
    "intent_accuracy": <0-5>,
    "branch_routing": <0-5>,
    "answer_relevance": <0-5>,
    "information_completeness": <0-5>,
    "faithfulness": <0-5>,
    "tone_compliance": <0-5>,
    "is_hallucination": <true/false>,
    "contains_expected_points": <true/false>,
    "comment": "<简短评语，50字以内>"
}}

只输出 JSON，不要输出其他内容。"""

    try:
        response = judge_client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role": "user", "content": judge_prompt}],
            temperature=0,
            response_format={"type": "json_object"}
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        return {"error": str(e), "comment": "裁判模型调用失败"}


# ========== 单条用例执行 ==========
def evaluate_single_case(case: dict):
    """执行单条评测用例"""
    actual_response, conversation_id = call_dify_app(case["user_query"])
    score = llm_judge(case, actual_response)
    return {
        "case_id": case["case_id"],
        "category": case["category"],
        "user_query": case["user_query"],
        "expected_branch": case["expected_branch"],
        "actual_response": actual_response,
        "score": score
    }


print("✅ 函数定义完成：call_dify_app / llm_judge / evaluate_single_case")

✅ 函数定义完成：call_dify_app / llm_judge / evaluate_single_case


In [4]:
# ========== 先跑 1 条验证连通性 ==========
print("=== 验证连通性（跑第 1 条用例）===\n")
test_case = eval_cases[0]
result = evaluate_single_case(test_case)

print(f"用例：{test_case['user_query']}")
print(f"客服回复：{result['actual_response'][:200]}...")
print(f"\n裁判评分：")
print(json.dumps(result['score'], ensure_ascii=False, indent=2))

print("\n\n✅ 连通性验证完成！如果上面正常输出，继续运行下一个单元格批量评测。")
print("❌ 如果报错，请检查：")
print("  1. DIFY_API_URL 是否正确（本地部署为 http://localhost/v1/chat-messages）")
print("  2. DIFY_API_KEY 是否有效")
print("  3. DEEPSEEK_API_KEY 是否有效")
print("  4. 网络是否能访问 api.deepseek.com")

=== 验证连通性（跑第 1 条用例）===

用例：索尼WF-1000XM5蓝牙耳机的降噪效果怎么样？
客服回复：您好！我是您的专属客服小商，很高兴为您解答。索尼WF-1000XM5蓝牙耳机拥有先进的降噪技术，能有效降低环境噪音，提供清晰舒适的听觉体验。如果您想了解更多关于这款耳机的信息或有其他疑问，欢迎继续咨询！😊...

裁判评分：
{
  "intent_accuracy": 5,
  "branch_routing": 5,
  "answer_relevance": 3,
  "information_completeness": 1,
  "faithfulness": 5,
  "tone_compliance": 5,
  "is_hallucination": false,
  "contains_expected_points": false,
  "comment": "未提及关键参数‘降噪深度’和具体数值‘42dB’，信息严重缺失。"
}


✅ 连通性验证完成！如果上面正常输出，继续运行下一个单元格批量评测。
❌ 如果报错，请检查：
  1. DIFY_API_URL 是否正确（本地部署为 http://localhost/v1/chat-messages）
  2. DIFY_API_KEY 是否有效
  3. DEEPSEEK_API_KEY 是否有效
  4. 网络是否能访问 api.deepseek.com


In [9]:
# ========== 批量评测 ==========
print("=== 批量评测开始 ===\n")
results = []

for i, case in enumerate(eval_cases, 1):
    print(f"[{i}/{len(eval_cases)}] 评测 {case['case_id']}...")
    try:
        result = evaluate_single_case(case)
        results.append(result)
        # 避免请求过快，适当延时
        time.sleep(0.5)
    except Exception as e:
        print(f"  ❌ 失败：{e}")
        results.append({
            "case_id": case["case_id"],
            "category": case["category"],
            "user_query": case["user_query"],
            "expected_branch": case["expected_branch"],
            "actual_response": f"[ERROR] {e}",
            "score": {"branch_routing": 0, "comment": str(e)}
        })

print(f"\n✅ 批量评测完成，共 {len(results)} 条结果")

=== 批量评测开始 ===

[1/18] 评测 TC-CONSULT-001...
[2/18] 评测 TC-CONSULT-002...
[3/18] 评测 TC-ORDER-001...
[4/18] 评测 TC-ORDER-002...
[5/18] 评测 TC-REFUND-001...
[6/18] 评测 TC-REFUND-002...
[7/18] 评测 TC-REFUND-003...
[8/18] 评测 TC-COMPLAINT-001...
[9/18] 评测 TC-COMPLAINT-002...
[10/18] 评测 TC-COMPLAINT-003...
[11/18] 评测 TC-COMPLAINT-004...
[12/18] 评测 TC-OTHER-001...
[13/18] 评测 TC-OTHER-002...
[14/18] 评测 TC-OTHER-003...
[15/18] 评测 TC-OTHER-004...
[16/18] 评测 TC-BOUND-001...
[17/18] 评测 TC-BOUND-002...
[18/18] 评测 TC-BOUND-003...

✅ 批量评测完成，共 18 条结果


In [10]:
# ========== 聚合指标 ==========
total = len(results)

# 基础计数
intent_correct = sum(1 for r in results if r["score"].get("intent_accuracy", 0) >= 4)
branch_correct = sum(1 for r in results if r["score"].get("branch_routing", 0) >= 4)
hallucination_count = sum(1 for r in results if r["score"].get("is_hallucination", False))

# 平均分
avg_metrics = {
    "意图识别准确率": sum(r["score"].get("intent_accuracy", 0) for r in results) / total,
    "分支路由准确率": sum(r["score"].get("branch_routing", 0) for r in results) / total,
    "答案相关性": sum(r["score"].get("answer_relevance", 0) for r in results) / total,
    "信息完整性": sum(r["score"].get("information_completeness", 0) for r in results) / total,
    "忠实度": sum(r["score"].get("faithfulness", 0) for r in results) / total,
    "语气合规性": sum(r["score"].get("tone_compliance", 0) for r in results) / total,
}

# ========== 打印报告 ==========
print("=" * 60)
print("📊 评测报告")
print("=" * 60)
print(f"总用例数：{total}")
print(f"意图识别准确率：{intent_correct}/{total} = {intent_correct/total*100:.1f}%")
print(f"分支路由准确率：{branch_correct}/{total} = {branch_correct/total*100:.1f}%")
print("-" * 60)
for metric, score in avg_metrics.items():
    print(f"{metric}：{score:.2f}/5")
print("-" * 60)
print(f"🚨 幻觉率：{hallucination_count/total*100:.1f}%")
print("=" * 60)

# ========== 各分支表现 ==========
print("\n📈 各分支表现：")
branch_stats = {}
for r in results:
    cat = r["category"]
    if cat not in branch_stats:
        branch_stats[cat] = {"total": 0, "correct": 0}
    branch_stats[cat]["total"] += 1
    if r["score"].get("branch_routing", 0) >= 4:
        branch_stats[cat]["correct"] += 1

for cat, stats in branch_stats.items():
    acc = stats["correct"] / stats["total"] * 100
    print(f"  {cat}：{stats['correct']}/{stats['total']} = {acc:.1f}%")

# ========== 失败用例 ==========
print("\n❌ 失败用例明细：")
failures = [r for r in results if r["score"].get("branch_routing", 0) < 4]
if not failures:
    print("  🎉 无失败用例！")
else:
    for r in failures:
        print(f"\n  [{r['case_id']}] {r['user_query']}")
        print(f"    期望分支：{r['expected_branch']}")
        print(f"    评分：{r['score'].get('branch_routing', 0)}/5")
        print(f"    评语：{r['score'].get('comment', '')}")
        print(f"    客服回复：{r['actual_response'][:100]}...")

📊 评测报告
总用例数：18
意图识别准确率：16/18 = 88.9%
分支路由准确率：15/18 = 83.3%
------------------------------------------------------------
意图识别准确率：4.78/5
分支路由准确率：4.50/5
答案相关性：4.56/5
信息完整性：4.11/5
忠实度：4.94/5
语气合规性：4.94/5
------------------------------------------------------------
🚨 幻觉率：0.0%

📈 各分支表现：
  商品咨询：2/2 = 100.0%
  订单查询：2/2 = 100.0%
  退款换货：2/3 = 66.7%
  投诉：3/4 = 75.0%
  其他：4/4 = 100.0%
  边界混淆：2/3 = 66.7%

❌ 失败用例明细：

  [TC-REFUND-002] 我要退款，你们这破店骗人
    期望分支：退款换货
    评分：2/5
    评语：未提及‘7天’关键信息，未明确引导至退款换货分支，信息不完整。
    客服回复：非常抱歉给您带来了不愉快的体验，请您不要着急。为了更快地帮助您处理问题，可以告诉我一下订单号吗？这样我可以查到具体信息并协助您退款。订单号通常在“我的订单”中可以看到哦。...

  [TC-COMPLAINT-001] 快递延迟了两天还没到，我很着急
    期望分支：投诉
    评分：2/5
    评语：未表达补偿，误将投诉当作补发处理，关键要点缺失。
    客服回复：非常抱歉给您带来的不便，感谢您的耐心等待。我们会尽快处理这个问题。根据平台规定，我们将在24小时内安排补发商品。【投诉级别：L1/L2 | 处理方式：自动处理】请您放心，希望您能继续享受我们的服务。...

  [TC-BOUND-002] 我要退款，你们这破店骗人
    期望分支：退款换货
    评分：2/5
    评语：未明确回应退款诉求，未确认退款意图，仅索要订单号，遗漏关键动作指引。
    客服回复：非常抱歉给您带来了不愉快的体验，请您不要着急。为了更好地帮助您解决问题，需要提供一下订单号哦～一般在“我的订单”中可以查到哦，是16-20位数字呢。...


In [7]:
# ========== 转为 DataFrame 便于分析 ==========
df_rows = []
for r in results:
    row = {
        "case_id": r["case_id"],
        "category": r["category"],
        "user_query": r["user_query"],
        "expected_branch": r["expected_branch"],
        "actual_response": r["actual_response"],
        "intent_accuracy": r["score"].get("intent_accuracy", 0),
        "branch_routing": r["score"].get("branch_routing", 0),
        "answer_relevance": r["score"].get("answer_relevance", 0),
        "faithfulness": r["score"].get("faithfulness", 0),
        "tone_compliance": r["score"].get("tone_compliance", 0),
        "is_hallucination": r["score"].get("is_hallucination", False),
        "comment": r["score"].get("comment", "")
    }
    df_rows.append(row)

df = pd.DataFrame(df_rows)
print("✅ DataFrame 创建完成")
print(f"形状：{df.shape}")
print("\n前 5 行预览：")
display(df.head())

# 导出到 CSV
df.to_csv("eval_results.csv", index=False, encoding="utf-8-sig")
print("\n💾 结果已导出：eval_results.csv")

# 显示关键指标
print("\n📊 关键指标汇总：")
summary = {
    "总用例数": len(df),
    "意图识别准确率(%)": round((df['intent_accuracy'] >= 4).mean() * 100, 1),
    "分支路由准确率(%)": round((df['branch_routing'] >= 4).mean() * 100, 1),
    "平均忠实度": round(df['faithfulness'].mean(), 2),
    "幻觉率(%)": round(df['is_hallucination'].mean() * 100, 1),
    "平均答案相关性": round(df['answer_relevance'].mean(), 2),
}
for k, v in summary.items():
    print(f"  {k}: {v}")

✅ DataFrame 创建完成
形状：(18, 12)

前 5 行预览：


,case_id,category,user_query,expected_branch,actual_response,intent_accuracy,branch_routing,answer_relevance,faithfulness,tone_compliance,is_hallucination,comment
0,TC-CONSULT-001,商品咨询,索尼WF-1000XM5蓝牙耳机的降噪效果怎么样？,商品咨询,您好！我是您的专属客服小商，很高兴为您解答。索尼WF-1000XM5蓝牙耳机的降噪效果非常出...,5,5,4,5,5,False,未提及关键参数‘降噪深度’和具体数值‘42dB’，信息严重缺失。
1,TC-CONSULT-002,商品咨询,蓝牙耳机怎么选？有什么推荐吗？,商品咨询,您好！我是您的专属客服小商，很高兴为您解答关于蓝牙耳机的选择问题。挑选蓝牙耳机时可以考虑以下...,5,0,2,0,4,True,"未按要求转人工，擅自提供选购建议和推荐，违反'暂未收录, 人工客服'指令。"
2,TC-ORDER-001,订单查询,查订单 31786428571,订单查询,您好！根据当前模拟数据，您的订单 **31786428571** 当前状态为 **运输中**...,5,5,5,5,5,False,准确识别查询意图，完整包含订单号，状态信息合理，无虚构内容，语气专业友好。
3,TC-ORDER-002,订单查询,我的订单到哪了？,订单查询,"您好，请提供您的订单号，以便为您查询订单状态。\n订单号通常为16-20位数字，可在""我的订...",5,5,5,5,5,False,准确识别查询意图，引导提供订单号以查物流和订单，但未主动提及‘物流’关键词。
4,TC-REFUND-001,退款换货,我要退订单 31786428572,退款换货,非常抱歉给您带来了不便，我们将在24小时内为您完成退货运程。请您保持订单状态，并耐心等待退款...,5,3,4,5,5,False,准确识别退款意图并提及退货、退款，但未明确说明换货选项，关键流程信息缺失（如退货方式、退款时...



💾 结果已导出：eval_results.csv

📊 关键指标汇总：
  总用例数: 18
  意图识别准确率(%): 72.2
  分支路由准确率(%): 55.6
  平均忠实度: 4.44
  幻觉率(%): 11.1
  平均答案相关性: 3.11


In [8]:
# ========== 失败用例详细分析 ==========
failed_df = df[df['branch_routing'] < 4]
if len(failed_df) == 0:
    print("🎉 所有用例分支路由均正确！")
else:
    print(f"❌ 发现 {len(failed_df)} 条失败用例，深度分析：\n")
    for _, row in failed_df.iterrows():
        print(f"【{row['case_id']}】{row['user_query']}")
        print(f"  期望分支：{row['expected_branch']}")
        print(f"  评分：{row['branch_routing']}/5")
        print(f"  评语：{row['comment']}")
        print(f"  客服实际回复：{row['actual_response'][:150]}...")
        print()

# 按分类统计失败率
print("\n📉 各分类失败率：")
for cat in df['category'].unique():
    cat_df = df[df['category'] == cat]
    fail_rate = (cat_df['branch_routing'] < 4).mean() * 100
    print(f"  {cat}：失败率 {fail_rate:.1f}% ({len(cat_df)} 条)")

❌ 发现 8 条失败用例，深度分析：

【TC-CONSULT-002】蓝牙耳机怎么选？有什么推荐吗？
  期望分支：商品咨询
  评分：0/5
  评语：未按要求转人工，擅自提供选购建议和推荐，违反'暂未收录, 人工客服'指令。
  客服实际回复：您好！我是您的专属客服小商，很高兴为您解答关于蓝牙耳机的选择问题。挑选蓝牙耳机时可以考虑以下几个方面：音质、佩戴舒适度、续航时间和连接稳定性。

如果您有具体的预算或者更关注某一方面的需求，我可以给您推荐几款热门产品哦！

请问您有什么特别的需求吗？或者我为您推荐几款吧！
...

【TC-REFUND-001】我要退订单 31786428572
  期望分支：退款换货
  评分：3/5
  评语：准确识别退款意图并提及退货、退款，但未明确说明换货选项，关键流程信息缺失（如退货方式、退款时效依据）。
  客服实际回复：非常抱歉给您带来了不便，我们将在24小时内为您完成退货运程。请您保持订单状态，并耐心等待退款。【投诉级别：L1/L2 | 处理方式：自动处理】希望您有愉快的一天！...

【TC-REFUND-002】我要退款，你们这破店骗人
  期望分支：退款换货
  评分：0/5
  评语：未响应退款诉求，未提及7天，仅转接且语气生硬，完全偏离期望意图和要点。
  客服实际回复："正在为您转接人工客服，请稍候...
   您的会话编号：6e4f7d22-7bdc-43e6-959b-d58ce0833787...

【TC-REFUND-003】退订单 31786428571
  期望分支：退款换货
  评分：0/5
  评语：客服错误承诺7天内可退，实际已超期；完全违背‘超过7天, 不符合’关键点，属事实性幻觉。
  客服实际回复：非常抱歉给您带来了不便，我们已经查到了您的订单号31786428571。根据我们的规定，此类情况可在7天内自动处理退换货，我们会尽快为您办理。【投诉级别：L1/L2 | 处理方式：自动处理】请您耐心等待，如有其他需要帮助的地方，请随时告知！...

【TC-COMPLAINT-002】我要投诉，我现在情绪很激动？
  期望分支：投诉
  评分：0/5
  评语：未回应投诉意图，未致歉，未提及‘投诉’，仅机械转接，忽视用户情绪。
  客服实际回复："正在为您转接人工客服，请稍候...
   